# California EV charging infrastructure - scrape + per-year reconstruction

This notebook pulls the full California public electric-vehicle station list from
the AFDC Station Locator API and then reconstructs a **per-year** view of the
charging network for 2010-2025.

**Why reconstruct instead of download one file per year?**  The API only serves
the *current* roster of existing stations - it does not expose historical
year-by-year snapshots.  Every station record, however, carries an `open_date`.
A station that opened on/before 31 Dec of year *Y* was therefore part of the
network in year *Y*.  Counting stations (and their charging ports) with
`open_date <= Y-12-31` yields the cumulative number of chargers active in each
county in each year - exactly what the county-year panel needs.  This is the same
source as the 2025 data, so the series is internally consistent.

In [17]:
import os
import requests
import pandas as pd

In [18]:
# Register a free API key at developer.nlr.gov and paste it here to access the
# AFDC alternative-fuel (electric) charging-station data for California.

API_KEY = "bgmyMYWzEYXXacdQwmoYAwdy7VtKwN5l2rKLTOKM"
url = "https://developer.nlr.gov/api/alt-fuel-stations/v1.json"
params = {
    "api_key": API_KEY,
    "fuel_type": "ELEC",
    "state": "CA",
    "status": "E",
    "access": "public",
    "limit": "all",
}

In [19]:
r = requests.get(url, params=params)
r.raise_for_status()

In [20]:
stations = pd.DataFrame(r.json()["fuel_stations"])

In [21]:
# Display the first few rows of the DataFrame to understand the structure of the data.
stations.head()

,access_code,access_days_time,access_detail_code,cards_accepted,date_last_confirmed,expected_date,fuel_type_code,groups_with_access_code,id,maximum_vehicle_class,...,rd_max_biodiesel_level,nps_unit_name,access_days_time_fr,intersection_directions_fr,bd_blends_fr,groups_with_access_code_fr,ev_pricing_fr,ev_charging_units,ev_network_ids,federal_agency
0,public,5:30am-9pm; pay lot,None,None,2025-12-12,None,ELEC,Public,1523,LD,...,None,None,None,None,None,Public,None,"[{'network': 'Non-Networked', 'connectors': {'...",NaN,NaN
1,public,24 hours daily,None,None,2024-08-15,None,ELEC,Public,6355,None,...,None,None,None,None,None,Public,None,"[{'network': 'Non-Networked', 'connectors': {'...",NaN,NaN
2,public,Dealership business hours; customer use only,CALL,None,2025-06-06,None,ELEC,Public - Call ahead,6405,MD,...,None,None,None,None,None,Public - Appeler à l'avance,None,"[{'network': 'Non-Networked', 'connectors': {'...",NaN,NaN
3,public,6am-12am daily,None,None,2023-09-14,None,ELEC,Public,6425,LD,...,None,None,None,None,None,Public,None,"[{'network': 'Non-Networked', 'connectors': {'...",NaN,NaN
4,public,24 hours daily; pay lot,None,CREDIT,2025-06-06,None,ELEC,Public,6505,LD,...,None,None,None,None,None,Public,None,"[{'network': 'Non-Networked', 'connectors': {'...",NaN,NaN


In [22]:
# Number of stations currently in the dataset (the 2025 snapshot).
len(stations)

19422

In [23]:
# Save the full current snapshot for reference / further cleaning.
stations.to_csv("ev_charging_stations_ca.csv", index=False)

## Reconstruct a per-year view of the network (2010-2025)

For each calendar year we keep every station that had already opened by 31 Dec of
that year (`open_date <= Y-12-31`).  We retain the location (`latitude`,
`longitude`) so the SQL step can map each station to a county, and the three EVSE
port counts so chargers can be tallied by type.  One CSV per year is written to
`ev_stations_by_year/`, mirroring the per-year file convention used elsewhere in
the `data/` tree (e.g. the air-quality files).

In [24]:
# Columns the downstream cleaning / panel build needs.
KEEP = [
    "open_date",
    "latitude",
    "longitude",
    "ev_level1_evse_num",
    "ev_level2_evse_num",
    "ev_dc_fast_num",
]

ev = stations[KEEP].copy()
ev["open_date"] = pd.to_datetime(ev["open_date"], errors="coerce")
for c in ["latitude", "longitude"]:
    ev[c] = pd.to_numeric(ev[c], errors="coerce")
for c in ["ev_level1_evse_num", "ev_level2_evse_num", "ev_dc_fast_num"]:
    ev[c] = pd.to_numeric(ev[c], errors="coerce").astype("Int64")

# Stations with no open_date cannot be placed in time; drop them from the
# year-by-year reconstruction (they are a tiny share of the roster).
ev = ev[ev["open_date"].notna()].copy()
print(f"{len(ev):,} stations with a usable open_date")

19,393 stations with a usable open_date


In [25]:
OUT_DIR = "ev_stations_by_year"
os.makedirs(OUT_DIR, exist_ok=True)

FIRST_YEAR, LAST_YEAR = 2010, 2025
summary = []
for yr in range(FIRST_YEAR, LAST_YEAR + 1):
    cutoff = pd.Timestamp(f"{yr}-12-31")
    snap = ev[ev["open_date"] <= cutoff].copy()
    snap.insert(0, "snapshot_year", yr)
    snap["open_date"] = snap["open_date"].dt.strftime("%Y-%m-%d")
    dst = os.path.join(OUT_DIR, f"ev_stations_ca_{yr}.csv")
    snap.to_csv(dst, index=False)
    summary.append((yr, len(snap), int(snap["ev_dc_fast_num"].fillna(0).sum())))

summary = pd.DataFrame(summary, columns=["year", "stations", "dcfc_ports"])
print(summary.to_string(index=False))

 year  stations  dcfc_ports
 2010        66           6
 2011       108          28
 2012       177          52
 2013       276         249
 2014       427         370
 2015       699         650
 2016       901         790
 2017      1092         826
 2018      1412        1849
 2019      1744        2790
 2020      3959        4395
 2021      8904        6100
 2022     11069        7927
 2023     13525       10256
 2024     15664       13178
 2025     18213       16688


The `stations` and `dcfc_ports` columns above should increase monotonically with
the year - confirming the network grows over time rather than being held constant.
These per-year files are the input to `00_prepare_xlsx.py`, which consolidates
them into the single `ev_stations_clean.csv` consumed by `01_build_panel.sql`.